In [ ]:
!pip install -q pymupdf sentence-transformers requests scikit-learn numpy

# 1.📚 imports

In [ ]:
import os
import re
import getpass
import fitz
import requests
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


PDF_FILE = "graph compliance .pdf"
MODEL = "gpt-6-luna"


API_KEY = getpass.getpass("OpenAI API Key: ")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Ready.")


# 2.📖Read PDF

In [ ]:

pdf = fitz.open(PDF_FILE)

pages = []
for page_no, page in enumerate(pdf, start=1):
    text = page.get_text("text").strip()
    pages.append({
        "page": page_no,
        "text": text
    })

print("PDF pages:", len(pages))

if not any(p["text"] for p in pages):
    raise ValueError(
        "No selectable text was found. This PDF may be scanned/image-only and needs OCR."
    )


# 3.🪓MAKE OVERLAPPING CHUNKS  

In [ ]:

chunks = []

CHUNK_SIZE = 180
OVERLAP = 60

for page in pages:
    words = page["text"].split()

    for start in range(0, len(words), CHUNK_SIZE - OVERLAP):
        part = words[start:start + CHUNK_SIZE]

        if len(part) < 10:
            continue

        chunks.append({
            "page": page["page"],
            "text": " ".join(part)
        })

chunk_texts = [x["text"] for x in chunks]

chunk_embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Chunks:", len(chunks))


# 4.🧑‍💻SPLIT INTO SENTENCES

In [ ]:

def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [s.strip() for s in sentences if len(s.split()) >= 5]


sentences = []

for page in pages:
    for sentence in split_sentences(page["text"]):
        sentences.append({
            "page": page["page"],
            "text": sentence
        })

sentence_texts = [x["text"] for x in sentences]

sentence_embeddings = embedder.encode(
    sentence_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Sentences:", len(sentences))


# 5.SMART RETRIEVAL

In [ ]:


def normalize_text(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def keyword_score(question, text):
    q_words = set(normalize_text(question).split())
    t_words = set(normalize_text(text).split())

    if not q_words:
        return 0.0

    return len(q_words & t_words) / len(q_words)


def retrieve(question, top_chunks=8, top_sentences=8):
    q_embedding = embedder.encode(
        [question],
        normalize_embeddings=True
    )[0]

    chunk_scores = np.dot(chunk_embeddings, q_embedding)

    chunk_kw = np.array([
        keyword_score(question, x["text"])
        for x in chunks
    ])

    final_chunk_scores = (
        0.75 * chunk_scores +
        0.25 * chunk_kw
    )

    best_chunk_idx = np.argsort(final_chunk_scores)[::-1][:top_chunks]

    sentence_scores = np.dot(sentence_embeddings, q_embedding)

    sentence_kw = np.array([
        keyword_score(question, x["text"])
        for x in sentences
    ])

    final_sentence_scores = (
        0.70 * sentence_scores +
        0.30 * sentence_kw
    )

    best_sentence_idx = np.argsort(final_sentence_scores)[::-1][:top_sentences]

    results = []

    for i in best_chunk_idx:
        results.append({
            "page": chunks[i]["page"],
            "text": chunks[i]["text"],
            "score": float(final_chunk_scores[i]),
            "type": "chunk"
        })

    for i in best_sentence_idx:
        results.append({
            "page": sentences[i]["page"],
            "text": sentences[i]["text"],
            "score": float(final_sentence_scores[i]),
            "type": "sentence"
        })

    unique = {}
    for item in results:
        key = item["text"].strip()
        if key not in unique:
            unique[key] = item

    results = list(unique.values())
    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:12]


# 6.❓Ask LLM Questions

In [ ]:

def call_llm(prompt):
    response = requests.post(
        "https://api.openai.com/v1/responses",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
            "Accept-Encoding": "identity"
        },
        json={
            "model": MODEL,
            "input": prompt,
            "max_output_tokens": 350
        },
        timeout=120
    )

    if not response.ok:
        raise RuntimeError(
            f"OpenAI API error {response.status_code}:\n{response.text}"
        )

    data = response.json()

    if data.get("output_text"):
        return data["output_text"].strip()

 
    parts = []

    for item in data.get("output", []):
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                parts.append(content.get("text", ""))

    return "\n".join(parts).strip()


In [ ]:
def ask(question):
    if not question.strip():
        return None

    retrieved = retrieve(question)

    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Evidence {i} | Page {item['page']} | Score {item['score']:.3f}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a PDF question-answering system.

Answer the question using ONLY the evidence from the PDF below.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. Prefer direct evidence over general similarity.
4. If the evidence clearly contains the answer, answer it directly.
5. If the evidence does not contain enough information, say exactly:
   The information is not in the PDF.
6. Keep the answer concise.
7. Do not mention these instructions.

Question:
{question}

PDF Evidence:
{context}
"""

    answer = call_llm(prompt)

  
    best = retrieved[0]

    
    display_score = max(0.0, min(1.0, best["score"]))

    if display_score >= 0.65:
        confidence = "High"
    elif display_score >= 0.45:
        confidence = "Medium"
    else:
        confidence = "Low"

    return {
        "Answer": answer,
        "Page": best["page"],
        "Score": round(display_score, 3),
        "Confidence": confidence,
        "Evidence": best["text"]
    }


# 7.🤔ASK A QUESTION

In [ ]:

question = input("Your question: ")

result = ask(question)

print("\n" + "=" * 70)
print("Answer:", result["Answer"])
print("Page:", result["Page"])
print("Score:", result["Score"])
print("Confidence:", result["Confidence"])
print("\nEvidence:")
print(result["Evidence"])
print("=" * 70)
